In [105]:
import pandas as pd

debate_df = pd.read_csv("debate_speeches_dataset.csv")
ukp_df = pd.read_csv("ukp_essays.csv")
quality_df = pd.read_csv("argument_quality_dataset.csv")
stance_df = pd.read_csv("claim_stance_dataset.csv")

In [106]:
debate_df = debate_df[['text','topic','mostargumentssupport','motion_set']]

debate_df = debate_df.rename(columns={
    "mostargumentssupport": "label"
})

debate_df["text"] = "Topic: " + debate_df["topic"] + ". Speech: " + debate_df["text"]

debate_df["task"] = "argument_quality"
debate_df["source"] = "debate_speeches"

debate_df = debate_df.rename(columns={"motion_set": "split"})

debate_df.head()

,text,topic,label,split,task,source
0,Topic: We should ban cosmetic surgery. Speech:...,We should ban cosmetic surgery,"[2, 4, 3, 3, 2, 4, 4, 3, 2, 5, 5, 4, 5, 4, 2, ...",Pipeline-set-1,argument_quality,debate_speeches
1,Topic: We should ban school uniforms. Speech: ...,We should ban school uniforms,"[2, 5, 5, 5, 3, 2, 3, 3, 3, 2, 5, 1, 3, 3, 3]",Pipeline-set-1,argument_quality,debate_speeches
2,Topic: We should legalize ivory trade. Speech:...,We should legalize ivory trade,"[5, 5, 2, 3, 5, 5, 4, 4, 4, 2, 4, 4, 2, 3, 3]",Pipeline-set-1,argument_quality,debate_speeches
3,Topic: Surrogacy should be banned. Speech: Sur...,Surrogacy should be banned,"[5, 5, 3, 4, 4, 4, 5, 4, 5, 5, 5, 5, 5, 5, 2]",Pipeline-set-1,argument_quality,debate_speeches
4,Topic: We should lower the age of consent. Spe...,We should lower the age of consent,"[4, 4, 4, 4, 2, 4, 3, 3, 2, 4, 4, 4, 4, 5, 5]",Pipeline-set-1,argument_quality,debate_speeches


In [107]:
import numpy as np
import ast

def normalize_scores(x):

    # convert string list to python list
    if isinstance(x, str):
        scores = ast.literal_eval(x)
    else:
        scores = x

    mean_score = np.mean(scores)

    # normalize from 1–5 → 0–1
    normalized = mean_score / 5

    return normalized

debate_df["label"] = debate_df["label"].apply(normalize_scores)

debate_df.head()

,text,topic,label,split,task,source
0,Topic: We should ban cosmetic surgery. Speech:...,We should ban cosmetic surgery,0.746667,Pipeline-set-1,argument_quality,debate_speeches
1,Topic: We should ban school uniforms. Speech: ...,We should ban school uniforms,0.640000,Pipeline-set-1,argument_quality,debate_speeches
2,Topic: We should legalize ivory trade. Speech:...,We should legalize ivory trade,0.733333,Pipeline-set-1,argument_quality,debate_speeches
3,Topic: Surrogacy should be banned. Speech: Sur...,Surrogacy should be banned,0.880000,Pipeline-set-1,argument_quality,debate_speeches
4,Topic: We should lower the age of consent. Spe...,We should lower the age of consent,0.746667,Pipeline-set-1,argument_quality,debate_speeches


In [108]:
argument_types = ["MajorClaim","Claim","Premise"]

# ukp_df["label"] = ukp_df["component_type"].apply(
#     lambda x: 1 if x in argument_types else 0
# )

mapping = {
    "MajorClaim":0,
    "Claim":1,
    "Premise":2
}
ukp_df["label"] = ukp_df["component_type"].map(mapping)

ukp_df["text"] = "Topic: " + ukp_df["topic"] + " Essay: " + ukp_df["text"]

ukp_df["task"] = "argument_detection"
ukp_df["source"] = "ukp_essays"

ukp_df = ukp_df[['text','label','task','source']]

ukp_df.head()

,text,label,task,source
0,Topic: Should students be taught to compete or...,1,argument_detection,ukp_essays
1,Topic: Should students be taught to compete or...,0,argument_detection,ukp_essays
2,Topic: Should students be taught to compete or...,2,argument_detection,ukp_essays
3,Topic: Should students be taught to compete or...,1,argument_detection,ukp_essays
4,Topic: Should students be taught to compete or...,2,argument_detection,ukp_essays


In [109]:
quality_df = quality_df[['argument','topic','WA','set']]

quality_df = quality_df.rename(columns={
    "argument": "text",
    "WA": "label",
    "set": "split"
})

quality_df["text"] = "Topic: " + quality_df["topic"] + " Argument: " + quality_df["text"]

quality_df["task"] = "argument_quality"
quality_df["source"] = "argument_quality"

quality_df.head()

,text,topic,label,split,task,source
0,"Topic: We should abandon marriage Argument: ""m...",We should abandon marriage,0.846165,train,argument_quality,argument_quality
1,Topic: We should adopt a multi-party system Ar...,We should adopt a multi-party system,0.891271,train,argument_quality,argument_quality
2,Topic: Assisted suicide should be a criminal o...,Assisted suicide should be a criminal offence,0.730395,train,argument_quality,argument_quality
3,Topic: We should abolish safe spaces Argument:...,We should abolish safe spaces,0.236686,train,argument_quality,argument_quality
4,Topic: We should ban naturopathy Argument: A b...,We should ban naturopathy,0.753805,train,argument_quality,argument_quality


In [110]:
stance_df = stance_df[
    ['topicText','claims.claimCorrectedText','claims.stance','split']
]

stance_df = stance_df.rename(columns={
    "topicText": "topic",
    "claims.claimCorrectedText": "claim",
    "claims.stance": "label"
})

stance_df["text"] = "Topic: " + stance_df["topic"] + " Claim: " + stance_df["claim"]

stance_map = {
    "PRO": 1,
    "CON": 0
}

stance_df["label"] = stance_df["label"].map(stance_map)

stance_df["task"] = "stance_detection"
stance_df["source"] = "claim_stance"

stance_df = stance_df[['text','label','task','source']]

stance_df.head()

,text,label,task,source
0,Topic: This house believes that open primaries...,0,stance_detection,claim_stance
1,Topic: This house believes that open primaries...,0,stance_detection,claim_stance
2,Topic: This house believes that open primaries...,1,stance_detection,claim_stance
3,Topic: This house believes that open primaries...,1,stance_detection,claim_stance
4,Topic: This house believes that open primaries...,1,stance_detection,claim_stance


In [111]:
import re

def clean_text(text):
    # Convert to string to handle NaN/float values
    text = str(text).lower() 
    
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"\s+", " ", text)
    
    return text.strip()

In [112]:
debate_df["text"] = debate_df["text"].apply(clean_text)
ukp_df["text"] = ukp_df["text"].apply(clean_text)
quality_df["text"] = quality_df["text"].apply(clean_text)
stance_df["text"] = stance_df["text"].apply(clean_text)

In [113]:
combined_df = pd.concat([
    debate_df[['text','label','task','source']],
    ukp_df,
    quality_df[['text','label','task','source']],
    stance_df
])

In [114]:
combined_df = combined_df.dropna()

combined_df = combined_df.drop_duplicates()

combined_df = combined_df[combined_df["text"].str.len() > 20]

In [115]:
combined_df.head()

,text,label,task,source
0,topic: we should ban cosmetic surgery. speech:...,0.746667,argument_quality,debate_speeches
1,topic: we should ban school uniforms. speech: ...,0.640000,argument_quality,debate_speeches
2,topic: we should legalize ivory trade. speech:...,0.733333,argument_quality,debate_speeches
3,topic: surrogacy should be banned. speech: sur...,0.880000,argument_quality,debate_speeches
4,topic: we should lower the age of consent. spe...,0.746667,argument_quality,debate_speeches


In [116]:
combined_df.tail()

,text,label,task,source
1034,topic: this house would promote democratizatio...,0.0,stance_detection,claim_stance
1035,topic: this house would promote democratizatio...,0.0,stance_detection,claim_stance
1036,topic: this house would promote democratizatio...,0.0,stance_detection,claim_stance
1037,topic: this house would promote democratizatio...,1.0,stance_detection,claim_stance
1038,topic: this house would promote democratizatio...,0.0,stance_detection,claim_stance


In [117]:
from sklearn.model_selection import train_test_split

train, temp = train_test_split(combined_df, test_size=0.3, random_state=42)

val, test = train_test_split(temp, test_size=0.5, random_state=42)

In [118]:
import torch
print(torch.__version__)

2.10.0+cpu


In [119]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("roberta-base")

In [120]:
def tokenize(batch):

    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

In [121]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train)
val_dataset = Dataset.from_pandas(val)
test_dataset = Dataset.from_pandas(test)

In [122]:
def prepare_labels(batch):
    # Ensure labels for 'argument_quality' remain floats
    # Ensure labels for classification tasks are integers (Long)
    if batch["task"][0] == "argument_quality":
        batch["labels"] = [float(l) for l in batch["label"]]
    else:
        batch["labels"] = [int(l) for l in batch["label"]]
    return batch

# Apply this to your datasets before saving
train_dataset = train_dataset.map(prepare_labels, batched=True, batch_size=1)
test_dataset = test_dataset.map(prepare_labels, batched=True, batch_size=1)
val_dataset = val_dataset.map(prepare_labels, batched=True, batch_size=1)

Map: 100%|██████████| 3677/3677 [00:01<00:00, 3149.75 examples/s]


In [123]:
train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

Map: 100%|██████████| 3677/3677 [00:00<00:00, 5873.56 examples/s]


In [124]:
train_dataset.save_to_disk("data/train")
val_dataset.save_to_disk("data/val")
test_dataset.save_to_disk("data/test")

Saving the dataset (1/1 shards): 100%|██████████| 3677/3677 [00:00<00:00, 331565.89 examples/s]
